In [ ]:
## In this python notebook, geo-coded EXPOSOME data are extracted, 
## matched to the most recent version of the Dutch 6-digit Postcode (PC6) 
## and aggregated down to PC4 level to merge them later 
## with the PC4-level livability scores edited by the Dutch ministry of public affairs

In [ ]:
# -----------------------------------------------------------------------------
# Block 0 — imports for needed operations
# -----------------------------------------------------------------------------
from pathlib import Path

import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

import os
import numpy as np
import dill
import matplotlib.colors as mcolors

os.getcwd()
# -----------------------------------------------------------------------------

In [2]:
# Block 1 — where the data lives (set ROOT once for the secure environment)
# -----------------------------------------------------------------------------
# Point ROOT at the folder that holds LISS/, EXPOSOME/, MISC/.
ROOT = Path("../../../source")

## EXPOSOME Data path
EXPOSOME = ROOT / "EXPOSOME"

## paths and files
## noise: single file, .gpkg, was left out due to lacking documentation

## urbanicity: single file
urban_file = EXPOSOME / "Urban" / "UR_B15_XX_XX_15_v3.tif"

## food access in environment: several files
food_path = EXPOSOME / "Food"
food_files = [food_path / f for f in os.listdir(food_path)]

## air pollution: Several files but what we are most interested in average exposure in 2023
air_pollution_file = EXPOSOME / "AirPollution" / "NO2B25_AAV_XX_XX_23_v2.tif"

## buildings: several files, relevant are walkability at 1 km (WAL_B10_XX_XX_20_v3.tif), 
## light exposure at 300m, 500m, 1000m (LAN_B03_XX_XX_20_v2.tif, LAN_B05_XX_XX_20_v2.tif, LAN_B10_XX_XX_20_v2.tif)
## Vegetation indices (MVI and NDV), for each, take median value (mean, median and SD were available)
built_path = EXPOSOME / "Built"
walkability_files = [built_path / f for f in list(built_path.glob("WAL*"))]
light_files = [built_path / f for f in list(built_path.glob("LAN*"))]
MVI_ME_files = [built_path / f for f in list(built_path.glob("MVI_ME*"))]
NDV_ME_files = [built_path / f for f in list(built_path.glob("NDV_ME*"))]

## output path for saved variables: each PC4-variable combination gets one dataframe
output_PC4_path = Path("../../data/exposome_pc4_aggregated")

## combining all exposure files
all_exposures_files = [
    urban_file, 
    air_pollution_file,
    *food_files,
    *walkability_files,
    *light_files,
    *MVI_ME_files,
    *NDV_ME_files
]

# Engine for all geopandas reads (fiona is broken on the secure machine).
GEO_ENGINE = "pyogrio"

In [3]:
## naming the exposures, essentially assigning variable names
variable_names_exposures = 
["urban", ## urbanicity
 "average_no2_23", # air pollution (annual mean of 2023)
 "FastF_800m", "FastF_KD_800m", "Rest_800m", "RestKD_800m", "Superm_800m", "Superm_KD_800m", ## food variables
 "wal_b03_20", "wal_b05_20", "wal_b10_20", ## walkability within buffers of 300, 500 and 1000m in 2020
 "lan_b03_20", "lan_b05_20", "lan_b10_20", ## light exposure at night within buffers of 300, 500 and 1000m in 2020
  "MVI_median_1000m_20", "MVI_median_300m_20", "MVI_median_500m_20", ## MVI (soil-adjusted vegatation) data, Median within buffers 1000, 300, and 500m
  "NDV_median_1000m_20", "NDV_median_300m_20", "NDV_median_500m_20"] ## NDV data (normalized difference vegetation index), Median within buffers 1000, 300, and 500m


In [4]:
## checking if the number of EXPOSOME files matches the variable labels
len(all_exposures_files) == len(variable_names_exposures)

True

In [ ]:
## Part 1: loading in PC6 file and rendering its geometry

## coordinates need to be matched, this loads in PC6, its geometry and 
## associated administrative-demographic variables
PC6_POLY_GPKG = ROOT / "MISC/2025-cbs_pc6_2024_v1/cbs_pc6_2024_v1.gpkg"

## norming PC6 column to be sure it has correct format
def norm_pc6(s):
    # "1509 BM" -> "1509BM" so the LISS key matches the CBS postcode6 format.
    return s.astype(str).str.replace(" ", "", regex=False).str.upper()

## PC6 polygons (CBS): load, normalise the postcode column, ensure RD (EPSG:28992).
pc6_poly = gpd.read_file(PC6_POLY_GPKG, engine=GEO_ENGINE)
print("pc6_poly cols:", pc6_poly.columns.tolist()[:6], "...")

for cand in ["pc6", "postcode6", "PC6", "postcode", "Postcode6"]:
    if cand in pc6_poly.columns:
        pc6_poly = pc6_poly.rename(columns={cand: "pc6"})
        break
else:
    raise KeyError(f"No postcode column found in {pc6_poly.columns.tolist()}")

pc6_poly["pc6"] = norm_pc6(pc6_poly["pc6"])

## correct geocoding
if pc6_poly.crs is None or pc6_poly.crs.to_epsg() != 28992:
    pc6_poly = pc6_poly.to_crs(28992)
print("pc6_poly:", len(pc6_poly), "| CRS:", pc6_poly.crs)

## take a centroid on PC6 level
pc6_poly = gpd.GeoDataFrame(pc6_poly, geometry="geometry", crs=28992)

## count areas without PC6
n_unmatched = pc6_poly.geometry.isna().sum()
print(f"Unmatched PC6: {n_unmatched} / {len(pc6_poly)} ({n_unmatched / len(pc6_poly):.1%})")
pc6_poly = pc6_poly.dropna(subset=["geometry"]).copy()

## create centroid column
pc6_poly["centroid"] = pc6_poly.geometry.centroid

## already create PC4 column
pc6_poly["pc4"] = pc6_poly["pc6"].str[:4]


In [ ]:
## Part 2: Linkage with the geo-coded exposure data

## link PC6 grid data with exposure data, looping over all variables / files previously defined
## as interesting

## creating lists for the pc4 and pc6 dataframes so they are still available in the working environment later 
## and can be used for map creation
pc6_df_full = pc6_poly.copy(deep=False)
pc4_dfs = []

## looping over all defined exposures: 
## - assign correct names
## - open the exposure file
## - render geometry to src
## - project data into the PC6 raster
## - extract values and match them with the PC6 area
## - aggregate exposure to PC4 level
## - save specific outcome df containing PC4 and specific exposure (files are merged in ML script in R) 

for exposure in range(0, len(variable_names_exposures)):

    exposure_name = variable_names_exposures[exposure]
    filename_exposure = all_exposures_files[exposure]

    ## 1) loading in exposome data,
    ## confirm it opens and report its grid.
    with rasterio.open(filename_exposure) as src:
        raster_crs = src.crs
        data_exposure = src.read(1)
        transform = src.transform
        print(f" {exposure_name} raster — CRS: {src.crs}, res: {src.res}, dtype: {src.dtypes[0]}")
        nodata = src.nodata
        ## merge exposome data to PC6 data creating PC6-resolution exposure
        ## create working copy of pc6_poly
        cent_raster = gpd.GeoSeries(pc6_poly["centroid"], crs=28992).to_crs(raster_crs)
        coords = [(p.x, p.y) for p in cent_raster]
        pc6_poly_df = pc6_poly.copy(deep=False)
        pc6_poly_df[exposure_name] = [v[0] for v in src.sample(coords)]
        ## raster is EPSG:3035, so reproject the
        ## centroids into the raster CRS first, then read the values.
    
    ## aggregate (mean) to PC4 (format to merge with the livability scores)
    pc4_exposure = pc6_poly_df.groupby("pc4")[exposure_name].mean().reset_index()
    print(f"PC4 areas with {exposure_name} values: {len(pc4_exposure)}")
        
    ## save data into exposures_PC4 folder
    filename = "data_" + exposure_name + "_PC4.csv"
    filename_full = output_PC4_path / filename
    pc4_exposure.to_csv(filename_full, index=False)
    print(f" File saved to {filename_full}") 

    ## merging the PC6 poly df with the exposome vector
    pc6_df_full.merge(pc6_poly_df[['pc6', exposure_name]], on="pc6", how= "outer")

    ## append to list pf pc4 dfs
    pc4_dfs.append(pc4_exposure)

print("loop exposure linkage finished")

## saving full pc6 dataframe for map visualizations
pc6_all_exposures_file = output_PC4_path / "pc6_all_exposome.csv"
pc6_df_full.to_csv(pc6_all_exposures_file, index=False)

In [ ]:
## Part 3: Aggregating administrative-demographic data that are contained in the
## PC6_poly file, these will also be used in the predictive modeling

## columns that should not be included in the aggregation
exclude_columns = ['geometry', 'centroid', 'pc6', 'pc4']

# Function to replace values less than -9900 with NaN and calculate mean
def custom_mean(series):
    # Replace values < -9900 with NaN
    series[series < -9900] = np.nan
    # Calculate the mean, ignoring NaNs
    return series.mean(skipna=True)

# dataframe to store the administrative-demographic data
pc4_demo_infr = pd.DataFrame()

# Loop over all columns except the excluded ones and calculate the mean
for column in pc6_poly.columns:
    if column not in exclude_columns:
        # Use groupby to apply the custom_mean function to each group
        grouped_means = pc6_poly.groupby('pc4')[column].apply(custom_mean)
        
        # Assign these means to the new DataFrame with a descriptive column name
        pc4_demo_infr[f'{column}_mean_pc4'] = grouped_means



In [21]:
## saving data on demographics and infrastructure on pc4 level
# pc4_demo_infr = pc4_demo_infr.reset_index()
# display(pc4_demo_infr.head())
pc4_demo_infr_file = output_PC4_path / "pc4_demo_infr.csv"
pc4_demo_infr.to_csv(pc4_demo_infr_file, index=False)

In [ ]:
## eoF